In [ ]:
import os
import logging
import math
import numpy as np
import pandas as pd
import pyvista as pv
import open3d as o3d
from tqdm import tqdm
from joblib import Parallel, delayed
from numba import njit
import open3d.core as o3c

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# =============================================================================
# Global cache for voxel-clipped meshes
# =============================================================================
clipped_mesh_cache = {}

# =============================================================================
# 1) LOAD MESH
# =============================================================================
def load_mesh(mesh_file_path):
    """Load a mesh with PyVista and return a PolyData or UnstructuredGrid."""
    if not os.path.exists(mesh_file_path):
        raise FileNotFoundError(f"Mesh file not found: {mesh_file_path}")
    mesh = pv.read(mesh_file_path)
    logging.info(f"Loaded mesh {mesh_file_path}, area={mesh.area:.4f}")
    return mesh

# =============================================================================
# 2) BOUNDS & VOXELS & CLIPPING
# =============================================================================
def combined_bounds(meshA, meshB):
    """Return bounding box that encloses both meshes."""
    a = meshA.bounds
    b = meshB.bounds
    xmin = min(a[0], b[0])
    xmax = max(a[1], b[1])
    ymin = min(a[2], b[2])
    ymax = max(a[3], b[3])
    zmin = min(a[4], b[4])
    zmax = max(a[5], b[5])
    return (xmin, xmax, ymin, ymax, zmin, zmax)

def create_voxel_centers_from_bounds(bounds, voxel_size):
    """
    Create a 3D grid of voxel centers spaced by 'voxel_size' inside the bounding box.
    """
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    xs = np.arange(xmin, xmax, voxel_size)
    ys = np.arange(ymin, ymax, voxel_size)
    zs = np.arange(zmin, zmax, voxel_size)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')
    centers = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T
    centers += (voxel_size / 2.0)
    return centers

def clip_mesh_to_voxel(voxel_center, voxel_size, mesh):
    """
    Clip the PyVista mesh to the cube bounding the voxel.
    Return the clipped mesh.
    """
    cube = pv.Cube(
        center=voxel_center,
        x_length=voxel_size,
        y_length=voxel_size,
        z_length=voxel_size
    )
    clipped = mesh.clip_box(cube, invert=False)
    return clipped

def convert_pyvista_to_open3d(pv_mesh):
    """
    Convert a PyVista mesh to an Open3D TriangleMesh (None if empty).
    """
    if pv_mesh.n_cells == 0:
        return None
    if isinstance(pv_mesh, pv.UnstructuredGrid):
        pv_mesh = pv_mesh.extract_geometry()
    if not pv_mesh.is_all_triangles:
        pv_mesh = pv_mesh.triangulate()
    vertices = np.asarray(pv_mesh.points)
    faces = np.asarray(pv_mesh.faces.reshape((-1, 4))[:, 1:])
    if len(faces) == 0:
        return None
    mesh_o3d = o3d.geometry.TriangleMesh()
    mesh_o3d.vertices = o3d.utility.Vector3dVector(vertices)
    mesh_o3d.triangles = o3d.utility.Vector3iVector(faces)
    mesh_o3d.compute_vertex_normals()
    return mesh_o3d

def get_clipped_meshes(voxel_center, voxel_size, leaf_mesh, wood_mesh):
    """
    Return pre-clipped and converted (to Open3D) leaf and wood meshes for the voxel.
    Uses a global cache to avoid repeated clipping operations.
    """
    key = (
        round(voxel_center[0], 6),
        round(voxel_center[1], 6),
        round(voxel_center[2], 6),
        voxel_size
    )
    if key in clipped_mesh_cache:
        return clipped_mesh_cache[key]
    clip_leaf = clip_mesh_to_voxel(voxel_center, voxel_size, leaf_mesh)
    clip_wood = clip_mesh_to_voxel(voxel_center, voxel_size, wood_mesh)
    o3d_leaf = convert_pyvista_to_open3d(clip_leaf)
    o3d_wood = convert_pyvista_to_open3d(clip_wood)
    clipped_mesh_cache[key] = (o3d_leaf, o3d_wood)
    return o3d_leaf, o3d_wood

# =============================================================================
# 3) RAY GENERATION (6 faces)
# =============================================================================
def generate_face_rays_bottom(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_z = voxel_center[2] - 0.5 * voxel_size
    gx = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gy = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    XX, YY = np.meshgrid(gx, gy, indexing='xy')
    origins = np.column_stack([
        voxel_center[0] + XX.ravel(),
        voxel_center[1] + YY.ravel(),
        np.full(XX.size, face_z)
    ])
    directions = np.tile([0, 0, 1], (len(origins), 1))
    return origins, directions

def generate_face_rays_top(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_z = voxel_center[2] + 0.5 * voxel_size
    gx = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gy = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    XX, YY = np.meshgrid(gx, gy, indexing='xy')
    origins = np.column_stack([
        voxel_center[0] + XX.ravel(),
        voxel_center[1] + YY.ravel(),
        np.full(XX.size, face_z)
    ])
    directions = np.tile([0, 0, -1], (len(origins), 1))
    return origins, directions

def generate_side_rays_xplus(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_x = voxel_center[0] + 0.5 * voxel_size
    gy = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gz = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    YY, ZZ = np.meshgrid(gy, gz, indexing='xy')
    origins = np.column_stack([
        np.full(YY.size, face_x),
        voxel_center[1] + YY.ravel(),
        voxel_center[2] + ZZ.ravel()
    ])
    directions = np.tile([-1, 0, 0], (len(origins), 1))
    return origins, directions

def generate_side_rays_xminus(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_x = voxel_center[0] - 0.5 * voxel_size
    gy = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gz = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    YY, ZZ = np.meshgrid(gy, gz, indexing='xy')
    origins = np.column_stack([
        np.full(YY.size, face_x),
        voxel_center[1] + YY.ravel(),
        voxel_center[2] + ZZ.ravel()
    ])
    directions = np.tile([1, 0, 0], (len(origins), 1))
    return origins, directions

def generate_side_rays_yplus(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_y = voxel_center[1] + 0.5 * voxel_size
    gx = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gz = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    XX, ZZ = np.meshgrid(gx, gz, indexing='xy')
    origins = np.column_stack([
        voxel_center[0] + XX.ravel(),
        np.full(XX.size, face_y),
        voxel_center[2] + ZZ.ravel()
    ])
    directions = np.tile([0, -1, 0], (len(origins), 1))
    return origins, directions

def generate_side_rays_yminus(voxel_center, voxel_size, ray_spacing):
    import math
    expanded_size = voxel_size * math.sqrt(2)
    steps = int(np.ceil(expanded_size / ray_spacing))
    face_y = voxel_center[1] - 0.5 * voxel_size
    gx = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    gz = np.linspace(-expanded_size / 2, expanded_size / 2, steps)
    XX, ZZ = np.meshgrid(gx, gz, indexing='xy')
    origins = np.column_stack([
        voxel_center[0] + XX.ravel(),
        np.full(XX.size, face_y),
        voxel_center[2] + ZZ.ravel()
    ])
    directions = np.tile([0, 1, 0], (len(origins), 1))
    return origins, directions

# =============================================================================
# 4) ROTATION FUNCTIONS
# =============================================================================
def rotate_rays(origins, directions, angle_degs, voxel_center):
    """Rotate rays about the X-axis by angle_degs."""
    angle_rad = np.radians(angle_degs)
    R = np.array([
        [1, 0,               0],
        [0, np.cos(angle_rad), -np.sin(angle_rad)],
        [0, np.sin(angle_rad),  np.cos(angle_rad)]
    ])
    shifted = origins - voxel_center
    rotated_origins = (shifted @ R.T) + voxel_center
    rotated_dirs = directions @ R.T
    return rotated_origins, rotated_dirs

def rotate_rays_around_y(origins, directions, angle_degs, voxel_center):
    """Rotate rays about the Y-axis by angle_degs."""
    angle_rad = np.radians(angle_degs)
    R = np.array([
        [ np.cos(angle_rad),  0, np.sin(angle_rad)],
        [ 0,                  1, 0               ],
        [-np.sin(angle_rad),  0, np.cos(angle_rad)]
    ])
    shifted = origins - voxel_center
    rotated_origins = (shifted @ R.T) + voxel_center
    rotated_dirs = directions @ R.T
    return rotated_origins, rotated_dirs

# =============================================================================
# 5) RAY-BOX INTERSECTION & LEAF ANGLE DISTRIBUTIONS
# =============================================================================
def ray_box_intersection_vectorized(origins, directions, box_min, box_max, eps=1e-12):
    """
    Vectorized bounding box intersection: returns t_near, t_far for each ray.
    """
    safe_dir = np.where(
        np.abs(directions) < eps,
        np.where(directions >= 0, eps, -eps),
        directions
    )
    t1 = (box_min - origins) / safe_dir
    t2 = (box_max - origins) / safe_dir
    t_near = np.maximum.reduce(np.minimum(t1, t2), axis=1)
    t_far = np.minimum.reduce(np.maximum(t1, t2), axis=1)
    return t_near, t_far

def compute_LIAD_from_mesh(mesh_o3d, num_bins=18):
    """
    Compute area-weighted leaf inclination distribution.
    Returns: (bin centers, LIAD, mean angle)
    """
    if (mesh_o3d is None) or (len(mesh_o3d.triangles) == 0):
        return np.array([]), np.array([]), np.nan
    verts = np.asarray(mesh_o3d.vertices)
    tris = np.asarray(mesh_o3d.triangles)
    v0 = verts[tris[:, 1]] - verts[tris[:, 0]]
    v1 = verts[tris[:, 2]] - verts[tris[:, 0]]
    cross_prod = np.cross(v0, v1)
    areas = 0.5 * np.linalg.norm(cross_prod, axis=1)
    norms = np.linalg.norm(cross_prod, axis=1, keepdims=True)
    normals = np.divide(cross_prod, norms, where=(norms != 0))
    angle_facets = np.degrees(np.arccos(np.clip(normals[:, 2], -1, 1)))
    angle_facets = np.where(angle_facets > 90, 180 - angle_facets, angle_facets)
    mean_angle = angle_facets.mean() if angle_facets.size > 0 else np.nan
    bin_edges = np.linspace(0, 90, num_bins + 1)
    idx = np.digitize(angle_facets, bin_edges) - 1
    idx = np.clip(idx, 0, num_bins - 1)
    bin_counts = np.bincount(idx, weights=areas, minlength=num_bins)
    total_area = areas.sum()
    liad = bin_counts / total_area if total_area > 0 else np.zeros(num_bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return bin_centers, liad, mean_angle

def compute_G_function_binwise(viewing_angles, leaf_bin_centers, LIAD_values):
    """
    Compute G(angle) from the leaf angle distribution (LIAD), binwise integration.
    """
    total_lad = LIAD_values.sum()
    lad_norm = LIAD_values / total_lad if total_lad > 0 else LIAD_values.copy()
    viewing_angles = np.clip(viewing_angles, 0.0001, 89.9999)
    leaf_bin_centers = np.clip(leaf_bin_centers, 0.0001, 89.9999)
    theta_rad = np.radians(viewing_angles)
    theta_lrad = np.radians(leaf_bin_centers)
    cos_theta = np.cos(theta_rad)
    cot_theta = 1 / np.tan(theta_rad)
    cos_tl = np.cos(theta_lrad)
    cot_tl = 1 / np.tan(theta_lrad)
    cot_mesh = np.outer(cot_theta, cot_tl)
    cos_mesh = np.outer(cos_theta, cos_tl)
    G_mat = np.zeros_like(cot_mesh)
    mask_gt1 = (np.abs(cot_mesh) > 1)
    mask_le1 = ~mask_gt1
    G_mat[mask_gt1] = cos_mesh[mask_gt1]
    inside = np.clip(cot_mesh[mask_le1], -1, 1)
    psi = np.arccos(inside)
    factor = 1.0 + (2.0 / np.pi) * (np.tan(psi) - psi)
    G_mat[mask_le1] = cos_mesh[mask_le1] * factor
    G_values = G_mat @ lad_norm
    return G_values

# =============================================================================
# 6) EFFECTIVE PATH LENGTH (NUMBA)
# =============================================================================
@njit
def compute_efpl_array(d_arr, lambda_1):
    """
    Compute the effective length for each distance in d_arr:
       effective length = -ln(1 - lambda_1 * d) / lambda_1
    """
    result = np.empty(d_arr.shape, dtype=np.float64)
    for i in range(d_arr.shape[0]):
        d = d_arr[i]
        if d <= 0:
            result[i] = 0.0
        else:
            val = -math.log(1.0 - lambda_1 * d) / lambda_1
            if np.isfinite(val):
                result[i] = val
            else:
                result[i] = 0.0
    return result

# =============================================================================
# 7) METRIC FUNCTIONS
# =============================================================================
def compute_lad_metrics(
    hits_leaf, N, G_leaf, delta_bar, mean_z, mean_delta_e, var_delta_e, lambda_1
):
    """
    Compute LAD metrics from the leaf-only simulation.
    Uses the full path length delta_bar (mean of Ã) and the effective free path length (mean_delta_e from Z).
    """
    results = {
        'LAD_BL': np.nan,
        'LAD_BL_EPL': np.nan,
        'LAD_BL_UEPL': np.nan,
        'LAD_MCF': np.nan,
        'LAD_MCF_Corrected': np.nan
    }
    eps = 1e-9
    I_leaf = hits_leaf / float(N) if N > 0 else 0.0
    if (N > 0) and (0 < I_leaf < 1) and (delta_bar > eps) and (G_leaf > eps):
        results['LAD_BL'] = -math.log(1.0 - I_leaf) / (G_leaf * delta_bar)  ## Pimont et al. 2018, eq. 5
    # Compute bot unconditionally when N > 0 so it can be used later
    if N > 0:
        bot = math.log(2.0 * N + 2.0)
    else:
        bot = np.nan
        
    if (N > 0) and (0 < I_leaf < 1) and (mean_delta_e > eps):
        top = math.log(1 - I_leaf) + (I_leaf / (2.0 * N * (1 - I_leaf)))
        bot = math.log(2.0 * N + 2.0)
        if abs(bot) > eps:
            val_epl = -(1.0 / mean_delta_e) * top
            results['LAD_BL_EPL'] = val_epl 
    elif I_leaf == 1.0:
        results['LAD_BL_EPL'] = bot / mean_delta_e 
    lad_bl_epl_val = results['LAD_BL_EPL']  ## Pimont et al. 2018, eq. 25
    if (not np.isnan(lad_bl_epl_val) and lad_bl_epl_val > 0 and
        var_delta_e > eps and mean_delta_e > eps):
        a_e = (mean_delta_e ** 2) / var_delta_e
        inside = 1.0 - 2.0 * a_e * lad_bl_epl_val
        if inside >= 0.0:
            val_uepl = (1.0 / a_e) * (1.0 - math.sqrt(inside))
            results['LAD_BL_UEPL'] = val_uepl / G_leaf  ## Pimont et al. 2018, eq. 27
    if (not np.isnan(lad_bl_epl_val) and lad_bl_epl_val > 0 and (G_leaf > eps)):
        results['LAD_BL_EPL'] = lad_bl_epl_val / G_leaf
        
    if (mean_z > eps) and (G_leaf > eps):
        results['LAD_MCF'] = I_leaf / (G_leaf * mean_z)  ## Pimont et al. 2018, eq. 8
    if (lambda_1 > 0 and delta_bar > 0 and (0 < I_leaf < 1) and 
        (mean_z > eps) and (1 - lambda_1 * delta_bar) > 0):
        denom = math.log(1.0 - lambda_1 * delta_bar) * mean_z
        if abs(denom) > eps:
            val_corr = -1.0 * (lambda_1 * delta_bar * I_leaf) / denom
            results['LAD_MCF_Corrected'] = val_corr / G_leaf  ## Pimont et al. 2018, eq. 12
    return results

def compute_pad_metrics(
    hits_lw, hits_leaf, N, G_lw, delta_bar, sum_z_lw, sum_z_e_lw, sum_hits_z_e_leaf,
    sum_hits_z_e_lw, alpha, mean_delta_e, var_delta_e, lambda_1, leaf_fraction
):
    """
    Compute PAD metrics from the combined (leaf+wood) simulation.
    """
    results = {
        'PAD_BL': np.nan,
        'PAD_BL_EPL': np.nan,
        'PAD_BL_UEPL': np.nan,
        'PAD_MCF': np.nan,
        'PAD_MCF_Corrected': np.nan,
        'PAD_MLE_pimont_2018': np.nan,
        'LAD_MLE_pimont_2019': np.nan,
        'LAD_MLE_Soma_21': np.nan
    }
    eps = 1e-9
    I_lw = hits_lw / float(N) if N > 0 else 0.0
    if (N > 0) and (0 < I_lw < 1) and (delta_bar > eps) and (G_lw > eps):
        results['PAD_BL'] = -math.log(1.0 - I_lw) / (G_lw * delta_bar)
    # Compute bot unconditionally when N > 0 so it can be used later
    if N > 0:
        bot = math.log(2.0 * N + 2.0)
    else:
        bot = np.nan
    if (N > 0) and (0 < I_lw < 1) and (mean_delta_e > eps):
        top = math.log(1 - I_lw) + (I_lw / (2.0 * N * (1 - I_lw)))
        bot = math.log(2.0 * N + 2.0)
        if abs(bot) > eps:
            val_epl = -(1.0 / mean_delta_e) * top 
            results['PAD_BL_EPL'] = val_epl
    elif I_lw == 1.0:
        results['PAD_BL_EPL'] = bot / mean_delta_e
    pad_bl_epl_val = results['PAD_BL_EPL']
    if (not np.isnan(pad_bl_epl_val) and pad_bl_epl_val > 0 and
        var_delta_e > eps and mean_delta_e > eps):
        a_e = (mean_delta_e ** 2) / var_delta_e
        inside = 1.0 - 2.0 * a_e * pad_bl_epl_val
        if inside >= 0.0:
            val_uepl = (1.0 / a_e) * (1.0 - math.sqrt(inside))
            results['PAD_BL_UEPL'] = val_uepl / G_lw
    if (not np.isnan(pad_bl_epl_val) and pad_bl_epl_val > 0 and (G_lw > eps)):
        results['PAD_BL_EPL'] = pad_bl_epl_val / G_lw
        
    mean_z_lw = sum_z_lw / N if N > 0 else 0.0  
    if (mean_z_lw > eps) and (G_lw > eps):
        results['PAD_MCF'] = I_lw / (G_lw * mean_z_lw)
    if (lambda_1 > 0 and delta_bar > 0 and (0 < I_lw < 1) and 
        (mean_z_lw > eps) and (1 - lambda_1 * delta_bar) > 0):
        denom = math.log(1.0 - lambda_1 * delta_bar) * mean_z_lw
        if abs(denom) > eps:
            val_corr = -1.0 * (lambda_1 * delta_bar * I_lw) / denom
            results['PAD_MCF_Corrected'] = val_corr / G_lw
    leaf_fraction = hits_leaf / hits_lw if hits_lw > 0 else 0.0
    if (G_lw > eps) and (sum_z_e_lw > eps):
        bracket = hits_lw - (sum_hits_z_e_lw / sum_z_e_lw)
        results['PAD_MLE_pimont_2018'] = (alpha * leaf_fraction / (G_lw * sum_z_e_lw)) * bracket
    leaf_fraction = hits_leaf / hits_lw if hits_lw > 0 else 0.0
    if (G_lw > eps) and (sum_z_e_lw > eps):
        bracket = hits_lw - (sum_hits_z_e_leaf / sum_z_e_lw)
        results['LAD_MLE_pimont_2019'] = (alpha * leaf_fraction / (G_lw * sum_z_e_lw)) * bracket
    leaf_fraction = hits_leaf / hits_lw if hits_lw > 0 else 0.0
    if (G_lw > eps) and (sum_z_e_lw > eps):
        bracket = hits_lw - (sum_hits_z_e_leaf / sum_z_lw)
        results['LAD_MLE_Soma_21'] = (leaf_fraction / (G_lw * sum_z_lw)) * bracket
    return results

# =============================================================================
# 8) WOOD VOLUME PROPORTION PER VOXEL
# =============================================================================
def compute_wood_volume_in_voxel(wood_interior_points, voxel_center, voxel_size, small_voxel_size=0.02):
    """
    Approximate wood volume within the voxel by counting how many 'interior' points
    fall into the voxel, times the volume of each small voxel.
    """
    half = voxel_size * 0.5
    min_x = voxel_center[0] - half
    max_x = voxel_center[0] + half
    min_y = voxel_center[1] - half
    max_y = voxel_center[1] + half
    min_z = voxel_center[2] - half
    max_z = voxel_center[2] + half
    inside_mask = (
        (wood_interior_points[:, 0] >= min_x) &
        (wood_interior_points[:, 0] < max_x) &
        (wood_interior_points[:, 1] >= min_y) &
        (wood_interior_points[:, 1] < max_y) &
        (wood_interior_points[:, 2] >= min_z) &
        (wood_interior_points[:, 2] < max_z)
    )
    count_in = np.count_nonzero(inside_mask)
    small_vol = small_voxel_size ** 3
    return count_in * small_vol

# =============================================================================
# 9) SINGLE & COMBINED RAYCAST SIMULATIONS
# =============================================================================
def simulate_single_mesh_with_points(mesh_o3d, voxel_center, voxel_size, ray_origins, ray_dirs, lambda_1, face_label="", angle_deg=0, object_label=""):
    """
    Cast rays into a single mesh (leaf or wood).
    Return summary stats for path length, hits, plus per-ray detail.
    """
    hit_details = []
    box_min = voxel_center - 0.5 * voxel_size
    box_max = voxel_center + 0.5 * voxel_size
    t_near, t_far = ray_box_intersection_vectorized(ray_origins, ray_dirs, box_min, box_max)
    valid_mask = (t_near <= t_far) & (t_far >= 0)
    N = np.count_nonzero(valid_mask)
    if (mesh_o3d is None) or (len(mesh_o3d.triangles) == 0) or (N == 0):
        empty_stats = dict(
            N=0, n_hits=0, I=0.0,
            delta_bar=0.0, sum_delta=0.0, sum_z=0.0,
            mean_delta_e=0.0, var_delta_e=0.0, sum_z_e=0.0, sum_hits_z_e=0.0,
            mean_z=0.0
        )
        return empty_stats, hit_details
    delta_arr = np.zeros(len(ray_origins), dtype=float)
    delta_arr[valid_mask] = (t_far[valid_mask] - t_near[valid_mask])
    z_arr = np.zeros(len(ray_origins), dtype=float)
    scene = o3d.t.geometry.RaycastingScene()
    tri_t = o3d.t.geometry.TriangleMesh.from_legacy(mesh_o3d)
    geom_id = scene.add_triangles(tri_t)
    big_len = voxel_size * 3.0
    rays_np = np.hstack([ray_origins, ray_origins + ray_dirs * big_len])
    rays_t = o3c.Tensor(rays_np, dtype=o3c.float32)
    hits = scene.cast_rays(rays_t)
    hit_gids = hits["geometry_ids"].numpy()
    hit_dist = hits["t_hit"].numpy()
    hit_mask = ((hit_gids == geom_id) &
                valid_mask &
                (hit_dist >= t_near) & (hit_dist <= t_far))
    n_hits = np.count_nonzero(hit_mask)
    I = n_hits / N if N > 0 else 0.0
    z_arr[hit_mask] = (hit_dist[hit_mask] - t_near[hit_mask])
    no_hit_mask = valid_mask & (~hit_mask)
    z_arr[no_hit_mask] = delta_arr[no_hit_mask]
    efpl_delta = compute_efpl_array(delta_arr, lambda_1)
    efpl_free  = compute_efpl_array(z_arr, lambda_1)
    mean_efpl_delta = efpl_delta[valid_mask].mean() if N > 0 else 0.0
    mean_efpl_free  = efpl_free[valid_mask].mean() if N > 0 else 0.0
    var_efpl_free   = efpl_free[valid_mask].var(ddof=1) if np.count_nonzero(valid_mask) > 1 else 0.0
    sum_delta = delta_arr[valid_mask].sum() if N > 0 else 0.0
    delta_bar = delta_arr[valid_mask].mean() if N > 0 else 0.0
    sum_z = z_arr[valid_mask].sum() if N > 0 else 0.0
    mean_z = sum_z / N if N > 0 else 0.0
    stats = dict(
        N=N,
        n_hits=n_hits,
        I=I,
        delta_bar=delta_bar,
        sum_delta=sum_delta,
        sum_z=sum_z,
        mean_z=mean_z,
        mean_delta_e=mean_efpl_free,
        var_delta_e=var_efpl_free,
        sum_z_e=efpl_free[valid_mask].sum() if N > 0 else 0.0,
        sum_hits_z_e=efpl_free[hit_mask].sum() if np.any(hit_mask) else 0.0
    )
    for idx in np.where(hit_mask)[0]:
        ox, oy, oz = ray_origins[idx]
        dx, dy, dz = ray_dirs[idx]
        t_val = hit_dist[idx]
        t_entry = t_near[idx]
        entry_x = ox + dx * t_entry
        entry_y = oy + dy * t_entry
        entry_z = oz + dz * t_entry
        hx = ox + dx * t_val
        hy = oy + dy * t_val
        hz = oz + dz * t_val
        path_len = t_val - t_entry
        hit_details.append({
            "voxel_cx": voxel_center[0],
            "voxel_cy": voxel_center[1],
            "voxel_cz": voxel_center[2],
            "face": face_label,
            "angle_deg": angle_deg,
            "t_entry_hit": t_entry,
            "entry_point_x": entry_x,
            "entry_point_y": entry_y,
            "entry_point_z": entry_z,
            "hit_point_x": hx,
            "hit_point_y": hy,
            "hit_point_z": hz,
            "hit_length": path_len,
            "Object": object_label if object_label else "LeafOrSingle"
        })
    return stats, hit_details

def simulate_combined_mesh_with_points(leaf_o3d, wood_o3d, voxel_center, voxel_size, ray_origins, ray_dirs, lambda_1, face_label="", angle_deg=0):
    """
    Cast rays into combined (leaf+wood) geometry. Return stats + ray hit details.
    """
    hit_details = []
    has_leaf = (leaf_o3d is not None and len(leaf_o3d.triangles) > 0)
    has_wood = (wood_o3d is not None and len(wood_o3d.triangles) > 0)
    if not (has_leaf or has_wood):
        empty_stats = dict(
            N=0, n_hits=0, I=0.0,
            delta_bar=0.0, sum_delta=0.0, sum_z=0.0, mean_z=0.0,
            mean_delta_e=0.0, var_delta_e=0.0, sum_z_e=0.0, sum_hits_z_e=0.0
        )
        return empty_stats, hit_details
    box_min = voxel_center - 0.5 * voxel_size
    box_max = voxel_center + 0.5 * voxel_size
    t_near, t_far = ray_box_intersection_vectorized(ray_origins, ray_dirs, box_min, box_max)
    valid_mask = (t_near <= t_far) & (t_far >= 0)
    N = np.count_nonzero(valid_mask)
    if N == 0:
        empty_stats = dict(
            N=0, n_hits=0, I=0.0,
            delta_bar=0.0, sum_delta=0.0, sum_z=0.0, mean_z=0.0,
            mean_delta_e=0.0, var_delta_e=0.0, sum_z_e=0.0, sum_hits_z_e=0.0
        )
        return empty_stats, hit_details
    delta_arr = np.zeros(len(ray_origins), dtype=float)
    delta_arr[valid_mask] = (t_far[valid_mask] - t_near[valid_mask])
    scene = o3d.t.geometry.RaycastingScene()
    geom_leaf_id = None
    geom_wood_id = None
    if has_leaf:
        tri_leaf_t = o3d.t.geometry.TriangleMesh.from_legacy(leaf_o3d)
        geom_leaf_id = scene.add_triangles(tri_leaf_t)
    if has_wood:
        tri_wood_t = o3d.t.geometry.TriangleMesh.from_legacy(wood_o3d)
        geom_wood_id = scene.add_triangles(tri_wood_t)
    big_len = voxel_size * 3.0
    rays_np = np.hstack([ray_origins, ray_origins + ray_dirs * big_len])
    rays_t = o3c.Tensor(rays_np, dtype=o3c.float32)
    hits = scene.cast_rays(rays_t)
    hit_gids = hits["geometry_ids"].numpy()
    hit_dist = hits["t_hit"].numpy()
    dist_leaf = np.full(len(ray_origins), np.inf, dtype=float)
    dist_wood = np.full(len(ray_origins), np.inf, dtype=float)
    if geom_leaf_id is not None:
        leaf_mask = ((hit_gids == geom_leaf_id) & valid_mask &
                     (hit_dist >= t_near) & (hit_dist <= t_far))
        dist_leaf[leaf_mask] = hit_dist[leaf_mask]
    if geom_wood_id is not None:
        wood_mask = ((hit_gids == geom_wood_id) & valid_mask &
                     (hit_dist >= t_near) & (hit_dist <= t_far))
        dist_wood[wood_mask] = hit_dist[wood_mask]
    combined_dist = np.minimum(dist_leaf, dist_wood)
    hit_any_mask = np.isfinite(combined_dist) & (combined_dist < np.inf)
    n_hits = np.count_nonzero(hit_any_mask)
    I = n_hits / N if N > 0 else 0.0
    for idx in np.where(hit_any_mask)[0]:
        ox, oy, oz = ray_origins[idx]
        dx, dy, dz = ray_dirs[idx]
        t_entry = t_near[idx]
        entry_x = ox + dx*t_entry
        entry_y = oy + dy*t_entry
        entry_z = oz + dz*t_entry
        if dist_leaf[idx] < dist_wood[idx]:
            obj_label = "Leaf"
            t_val = dist_leaf[idx]
        else:
            obj_label = "Wood"
            t_val = dist_wood[idx]
        hx = ox + dx*t_val
        hy = oy + dy*t_val
        hz = oz + dz*t_val
        path_len = t_val - t_entry
        hit_details.append({
            "voxel_cx": voxel_center[0],
            "voxel_cy": voxel_center[1],
            "voxel_cz": voxel_center[2],
            "face": face_label,
            "angle_deg": angle_deg,
            "t_entry_hit": t_entry,
            "entry_point_x": entry_x,
            "entry_point_y": entry_y,
            "entry_point_z": entry_z,
            "hit_point_x": hx,
            "hit_point_y": hy,
            "hit_point_z": hz,
            "hit_length": path_len,
            "Object": obj_label
        })
    z_arr = np.zeros(len(ray_origins), dtype=float)
    z_arr[hit_any_mask] = (combined_dist[hit_any_mask] - t_near[hit_any_mask])
    no_hit_mask = valid_mask & (~hit_any_mask)
    z_arr[no_hit_mask] = delta_arr[no_hit_mask]
    sum_delta = delta_arr[valid_mask].sum() if N > 0 else 0.0
    delta_bar = delta_arr[valid_mask].mean() if N > 0 else 0.0
    sum_z = z_arr[valid_mask].sum() if N > 0 else 0.0
    mean_z = sum_z / N if N > 0 else 0.0
    efpl_delta = compute_efpl_array(delta_arr, lambda_1)
    efpl_free  = compute_efpl_array(z_arr, lambda_1)
    mean_efpl_delta = efpl_delta[valid_mask].mean() if N > 0 else 0.0
    mean_efpl_free  = efpl_free[valid_mask].mean() if N > 0 else 0.0
    var_efpl_free   = efpl_free[valid_mask].var(ddof=1) if np.count_nonzero(valid_mask) > 1 else 0.0
    stats = dict(
        N=N,
        n_hits=n_hits,
        I=I,
        delta_bar=delta_bar,
        sum_delta=sum_delta,
        sum_z=sum_z,
        mean_z=mean_z,
        mean_delta_e=mean_efpl_free,
        var_delta_e=var_efpl_free,
        sum_z_e=efpl_free[valid_mask].sum() if N > 0 else 0.0,
        sum_hits_z_e=efpl_free[hit_any_mask].sum() if np.any(hit_any_mask) else 0.0
    )
    return stats, hit_details

# =============================================================================
# 10) PROCESS VOXEL
# =============================================================================
def process_voxel_3way(voxel_center, voxel_size, leaf_mesh, wood_mesh, ray_spacing, angles, lambda_1, wood_interior_points):
    """
    Process a single voxel using the new LAD and PAD estimation.
    Returns a list of result rows (one per face/angle combination) and a list of per-ray hit details.
    """
    MIN_HITS = 10
    o3d_leaf, o3d_wood = get_clipped_meshes(voxel_center, voxel_size, leaf_mesh, wood_mesh)
    if (o3d_leaf is None or len(o3d_leaf.triangles) == 0) and (o3d_wood is None or len(o3d_wood.triangles) == 0):
        return [], []
    leaf_area = o3d_leaf.get_surface_area() if (o3d_leaf is not None and len(o3d_leaf.triangles) > 0) else 0.0
    wood_area = o3d_wood.get_surface_area() if (o3d_wood is not None and len(o3d_wood.triangles) > 0) else 0.0
    LAI_leaf = leaf_area / (voxel_size ** 2)
    LAI_lw   = (leaf_area + wood_area) / (voxel_size ** 2)
    LAD_ref  = leaf_area / (voxel_size ** 3) if voxel_size > 0 else np.nan
    PAD_ref  = (leaf_area + wood_area) / (voxel_size ** 3) if voxel_size > 0 else np.nan
    wood_volume_m3 = compute_wood_volume_in_voxel(wood_interior_points, voxel_center, voxel_size, small_voxel_size=0.02)
    voxel_vol = voxel_size ** 3
    alpha = (voxel_vol - wood_volume_m3) / voxel_vol
    bin_leaf, liad_leaf, _ = compute_LIAD_from_mesh(o3d_leaf)
    if (o3d_leaf is not None) and (o3d_wood is not None):
        combined_mesh = o3d_leaf + o3d_wood
    elif o3d_leaf is not None:
        combined_mesh = o3d_leaf
    elif o3d_wood is not None:
        combined_mesh = o3d_wood
    else:
        combined_mesh = None
    bin_lw, liad_lw, _ = compute_LIAD_from_mesh(combined_mesh)
    voxel_rows = []
    all_hits_in_this_voxel = []
    faces = [
        ("bottom", generate_face_rays_bottom),
        ("top",    generate_face_rays_top),
        ("xplus",  generate_side_rays_xplus),
        ("xminus", generate_side_rays_xminus),
        ("yplus",  generate_side_rays_yplus),
        ("yminus", generate_side_rays_yminus),
    ]
    sorted_angles = sorted(angles)
    default_stats = {
        'N': 0, 'n_hits': 0, 'I': 0.0,
        'delta_bar': 0.0, 'sum_delta': 0.0, 'sum_z': 0.0,
        'mean_delta_e': 0.0, 'var_delta_e': 0.0,
        'sum_z_e': 0.0, 'sum_hits_z_e': 0.0, 'mean_z': 0.0
    }
    # Set fixed G values before using them in CI calculations:
    G_leaf = 0.5
    G_lw = 0.5
    for (face_label, face_fn) in faces:
        origins_face, dirs_face = face_fn(voxel_center, voxel_size, ray_spacing)
        for angle_deg in sorted_angles:
            if face_label in ("xplus", "xminus"):
                rot_o, rot_d = rotate_rays_around_y(origins_face, dirs_face, angle_deg, voxel_center)
            else:
                rot_o, rot_d = rotate_rays(origins_face, dirs_face, angle_deg, voxel_center)
            leaf_data, leaf_hits = simulate_single_mesh_with_points(
                mesh_o3d=o3d_leaf,
                voxel_center=voxel_center, voxel_size=voxel_size,
                ray_origins=rot_o, ray_dirs=rot_d,
                lambda_1=lambda_1,
                face_label=face_label,
                angle_deg=angle_deg,
                object_label="Leaf"
            )
            lw_data, lw_hits = simulate_combined_mesh_with_points(
                leaf_o3d=o3d_leaf, wood_o3d=o3d_wood,
                voxel_center=voxel_center, voxel_size=voxel_size,
                ray_origins=rot_o, ray_dirs=rot_d,
                lambda_1=lambda_1,
                face_label=face_label,
                angle_deg=angle_deg
            )
            if leaf_data is None or not isinstance(leaf_data, dict):
                leaf_data = default_stats.copy()
            if lw_data is None or not isinstance(lw_data, dict):
                lw_data = default_stats.copy()
            all_hits_in_this_voxel.extend(lw_hits)
            G_leaf_est = np.nan
            if leaf_data.get('n_hits', 0) >= MIN_HITS and (hasattr(bin_leaf, "size") and bin_leaf.size > 0):
                Gvals_leaf = compute_G_function_binwise(
                    viewing_angles=np.array([angle_deg]),
                    leaf_bin_centers=bin_leaf,
                    LIAD_values=liad_leaf
                )
                if len(Gvals_leaf) > 0:
                    G_leaf_est = Gvals_leaf[0]
            G_lw_est = np.nan
            if lw_data.get('n_hits', 0) >= MIN_HITS and (hasattr(bin_lw, "size") and bin_lw.size > 0):
                Gvals_lw = compute_G_function_binwise(
                    viewing_angles=np.array([angle_deg]),
                    leaf_bin_centers=bin_lw,
                    LIAD_values=liad_lw
                )
                if len(Gvals_lw) > 0:
                    G_lw_est = Gvals_lw[0]
            CI_Leaf = np.nan
            if (LAI_leaf > 0) and (leaf_data['I'] < 1) and (G_leaf > 0):
                gf_leaf = 1 - leaf_data['I']
                if gf_leaf > 0:
                    CI_Leaf = -math.log(gf_leaf) / (G_leaf * LAI_leaf)
            CI_LW = np.nan
            if (LAI_lw > 0) and (lw_data['I'] < 1) and (G_lw > 0):
                gf_lw = 1 - lw_data['I']
                if gf_lw > 0:
                    CI_LW = -math.log(gf_lw) / (G_lw * LAI_lw)
            hits_leaf = leaf_data.get('n_hits', 0)
            hits_lw = lw_data.get('n_hits', 0)
            leaf_fraction = hits_leaf / hits_lw if hits_lw > 0 else 0.0
            lad_leaf = compute_lad_metrics(
                hits_leaf=leaf_data.get('n_hits', 0),
                N=leaf_data.get('N', 0),
                G_leaf=G_leaf,
                delta_bar=leaf_data.get('delta_bar', 0.0),
                mean_z=leaf_data.get('mean_z', 0.0),
                mean_delta_e=leaf_data.get('mean_delta_e', 0.0),
                var_delta_e=leaf_data.get('var_delta_e', 0.0),
                lambda_1=lambda_1
            )
            if lad_leaf is None or not isinstance(lad_leaf, dict):
                lad_leaf = {}
            pad_lw = compute_pad_metrics(
                    hits_lw=lw_data.get('n_hits', 0),
                    hits_leaf=leaf_data.get('n_hits', 0),
                    N=lw_data.get('N', 0),
                    G_lw=G_lw,
                    delta_bar=lw_data.get('delta_bar', 0.0),
                    sum_z_lw=lw_data.get('sum_z', 0.0),
                    sum_z_e_lw=lw_data.get('sum_z_e', 0.0),
                    sum_hits_z_e_leaf=leaf_data.get('sum_hits_z_e', 0.0),
                    sum_hits_z_e_lw=lw_data.get('sum_hits_z_e', 0.0),  # corrected keyword
                    alpha=alpha,
                    leaf_fraction=leaf_fraction,
                    mean_delta_e=lw_data.get('mean_delta_e', 0.0),
                    var_delta_e=lw_data.get('var_delta_e', 0.0),
                    lambda_1=lambda_1
                )

            if pad_lw is None or not isinstance(pad_lw, dict):
                pad_lw = {}
            lad_mle_pimont = pad_lw.get("LAD_MLE_pimont_2019", np.nan)
            lad_mle_soma   = pad_lw.get("LAD_MLE_Soma_21", np.nan)
            row = {
                "voxel_cx": voxel_center[0],
                "voxel_cy": voxel_center[1],
                "voxel_cz": voxel_center[2],
                "face": face_label,
                "angle_deg": angle_deg,
                "G_leaf_computed": G_leaf_est,
                "G_lw_computed": G_lw_est,
                "G_leaf": G_leaf,
                "G_lw": G_lw,
                "LAI_Leaf": LAI_leaf,
                "LAI_lw": LAI_lw,
                "LAD_ref": LAD_ref,
                "PAD_ref": PAD_ref,
                "LAD_BL": lad_leaf.get("LAD_BL", np.nan),
                "LAD_BL_EPL": lad_leaf.get("LAD_BL_EPL", np.nan),
                "LAD_BL_UEPL": lad_leaf.get("LAD_BL_UEPL", np.nan),
                "LAD_MCF": lad_leaf.get("LAD_MCF", np.nan),
                "LAD_MCF_Corr": lad_leaf.get("LAD_MCF_Corrected", np.nan),
                "PAD_BL": pad_lw.get("PAD_BL", np.nan),
                "PAD_BL_EPL": pad_lw.get("PAD_BL_EPL", np.nan),
                "PAD_BL_UEPL": pad_lw.get("PAD_BL_UEPL", np.nan),
                "PAD_MCF": pad_lw.get("PAD_MCF", np.nan),
                "PAD_MCF_Corr": pad_lw.get("PAD_MCF_Corrected", np.nan),
                "PAD_MLE_pimont_2018": pad_lw.get("PAD_MLE_pimont_2018", np.nan),
                "LAD_MLE_pimont_2019": lad_mle_pimont,
                "LAD_MLE_soma": lad_mle_soma,
                "CI_leaf": CI_Leaf,
                "CI_lw": CI_LW,
                "alpha": alpha,
                "leaf_fraction": leaf_fraction,
                "Total_number_of_rays": lw_data.get("N", np.nan),
                "sum_path_length": lw_data.get("sum_delta", np.nan)
            }
            voxel_rows.append(row)
    return voxel_rows, all_hits_in_this_voxel

# =============================================================================
# 11) BATCH PROCESSING
# =============================================================================
def process_voxel_batch(voxel_centers_batch, voxel_size, leaf_mesh, wood_mesh,
                        ray_spacing, angles, lambda_1, wood_interior_points):
    batch_rows = []
    batch_hits = []
    for vc in voxel_centers_batch:
        rows, hits = process_voxel_3way(
            voxel_center=vc,
            voxel_size=voxel_size,
            leaf_mesh=leaf_mesh,
            wood_mesh=wood_mesh,
            ray_spacing=ray_spacing,
            angles=angles,
            lambda_1=lambda_1,
            wood_interior_points=wood_interior_points
        )
        batch_rows.extend(rows)
        batch_hits.extend(hits)
    return batch_rows, batch_hits
# =============================================================================
# 12) MAIN FUNCTION (Loop Over Tree IDs)
# =============================================================================
def process_tree(tree_id):
    # Construct file paths based on tree ID
    leaf_path = f"/QRISdata/Q5866/uqrarya1/Ref_Trees/Ref_Trees/{tree_id}_Trees/Leaf/{tree_id}_LEAF.obj"
    wood_path = f"/QRISdata/Q5866/uqrarya1/Ref_Trees/Ref_Trees/{tree_id}_Trees/Wood/{tree_id}_WOOD.obj"
    wood_interior_path = f"/QRISdata/Q5866/DATASETS/Synthetic_Validation_Trees/wood/{tree_id}_WOOD_inside_voxels_size0.01_thresh4.txt"
    base_output_dir = f"/QRISdata/Q5866/uqrarya1/result_analysis/voxel_all_metrics_references"
    os.makedirs(base_output_dir, exist_ok=True)
   
    # Load meshes
    leaf_mesh = load_mesh(leaf_path)
    wood_mesh = load_mesh(wood_path)
    bounds = combined_bounds(leaf_mesh, wood_mesh)
    
    ray_spacing = 0.005
    angles = [0.00001, 10, 20, 30, 40, 50, 60, 70, 80, 89.99999]
    cross_section_area = 0.002456134 # Change from the list
    wood_interior_points = np.loadtxt(wood_interior_path)
    logging.info(f"Tree {tree_id}: Loaded {len(wood_interior_points)} small wood voxels for interior-volume approximation.")
    
    voxel_sizes = [0.2]
    n_jobs = 96
    chunk_size = 5000
    
    for voxel_size in voxel_sizes:
        logging.info(f"Tree {tree_id}: Processing for voxel size: {voxel_size}")
        vcenters = create_voxel_centers_from_bounds(bounds, voxel_size)
        logging.info(f"Tree {tree_id}: Number of voxel centers for voxel size {voxel_size}: {len(vcenters)}")
        voxel_vol = voxel_size**3
        lambda_1 = cross_section_area / voxel_vol
        voxel_batches = np.array_split(vcenters, max(1, len(vcenters)//chunk_size))
        logging.info(f"Tree {tree_id}: Processing in {len(voxel_batches)} batches using {n_jobs} jobs.")
        results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10)(
            delayed(process_voxel_batch)(
                batch, voxel_size, leaf_mesh, wood_mesh,
                ray_spacing, angles, lambda_1, wood_interior_points
            )
            for batch in tqdm(voxel_batches, desc=f"Tree {tree_id} - Batches (voxel size={voxel_size})")
        )
        all_voxel_rows = []
        all_hits = []
        for (voxel_rows, voxel_hits) in results:
            all_voxel_rows.extend(voxel_rows)
            all_hits.extend(voxel_hits)
        df = pd.DataFrame(all_voxel_rows)
        out_csv = os.path.join(base_output_dir, f"{tree_id}_all_details_voxel_size_{voxel_size}.csv")
        df.to_csv(out_csv, index=False)
        logging.info(f"Tree {tree_id}: Saved CSV for voxel size {voxel_size} => {out_csv}")
        # df_hits = pd.DataFrame(all_hits)
        # ml_csv = os.path.join(base_output_dir, f"{tree_id}_ML_Input_Voxel_Size_{voxel_size}.csv")
        # df_hits.to_csv(ml_csv, index=False)
        # logging.info(f"Tree {tree_id}: Saved ML-Input CSV for voxel size {voxel_size} => {ml_csv}")

# =============================================================================
# 12) MAIN FUNCTION
# =============================================================================
def main():
    # List of tree IDs to process
    tree_ids = ["003"]  # Add additional tree IDs as needed
    for tree_id in tree_ids:
        logging.info(f"Starting processing for tree: {tree_id}")
        process_tree(tree_id)
        logging.info(f"Finished processing for tree: {tree_id}")
    print("All done!")

if __name__ == "__main__":
    main()
